# Extract biomass features with local inputs and outputs

This workflow copies **only one shard's pending S1/S2 TIFFs** from Google Drive to `/content`, extracts features with **local reads and writes**, then publishes verified NPYs into the same Drive cache.

Use six shards by default so both raw TIFFs and feature outputs fit together on a roughly 190 GB runtime disk. Preparation measures the selected files and checks local space with an 8 GiB reserve. Drive is read once during staging; it is not read by the GPU extraction loop.

Run setup once. For each `SHARD_ID` from 1 through `NUM_SHARDS`, run **Prepare ? Extract ? Merge**. Merge clears only verified local copies so the next shard fits. The complete merged cache still needs approximately 382 GiB in Drive.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 1. Configuration
Change `SHARD_ID` to 2, 3, ?, 6 after successfully merging the preceding shard, then rerun this configuration cell and the three workflow cells. Keep `NUM_SHARDS` unchanged during a run. Increase it before starting if the preparation space check fails; use a new `WORK_DIR` for a different partitioning scheme.


In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_DIR = Path('/content/BiomassEstimation')
BRANCH = 'colab-feature-extraction'
DRIVE_INPUTS = Path('/content/drive/MyDrive/projects/Biomass/Biomassters/train_features/Volumes/Samsung_T5/BIOMASS/BioMasster_Dataset/v1/DrivenData/train_features')
DRIVE_OUTPUTS = Path('/content/drive/MyDrive/projects/Biomass/Biomassters/extracted_features')
DRIVE_CHECKPOINT = Path('/content/drive/MyDrive/projects/Biomass/modelo_best.pth')
CHECKPOINT = Path('/content/modelo_best.pth')
WORK_DIR = Path('/content/biomass_feature_work')

NUM_SHARDS = 6
SHARD_ID = 1
BATCH_SIZE = 32
DEVICE = 'cuda'
NUM_WORKERS = 2
COPY_WORKERS = 4
RESERVE_GIB = 8
MAX_SAMPLES = 0  # Set 12 for a small extraction test; use a separate WORK_DIR for that preview.

PLAN = WORK_DIR / f'shard_{SHARD_ID:03d}_of_{NUM_SHARDS:03d}' / 'plan.json'
WORKFLOW = REPO_DIR / 'src' / 'colab_feature_workflow.py'
print(f'Shard {SHARD_ID}/{NUM_SHARDS}: {PLAN}')


## 2. Fetch the workflow branch
This does not reset or discard edits in an existing checkout. If Git reports local changes, resolve them before continuing.


In [ ]:
if not (REPO_DIR / '.git').is_dir():
    subprocess.run(['git', 'clone', '--branch', BRANCH,
                    'https://github.com/huzaifa-jawad367/BiomassEstimation.git', str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'switch', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', BRANCH], check=True)


In [ ]:
%pip install -r "{REPO_DIR}/requirements-colab.txt"


If installation asks for a runtime restart, restart and rerun mounting/configuration before continuing. Enable a GPU runtime. The checkpoint is trusted and contains serialized model objects.


In [ ]:
import shutil

if not CHECKPOINT.is_file():
    if not DRIVE_CHECKPOINT.is_file():
        raise FileNotFoundError(f'Place the checkpoint at {DRIVE_CHECKPOINT} or {CHECKPOINT}')
    shutil.copy2(DRIVE_CHECKPOINT, CHECKPOINT)

subprocess.run(['nvidia-smi'], check=True)
subprocess.run(['df', '-h', '/content'], check=True)


## 3. Prepare: copy only this shard's required raw inputs locally
This phase partitions the full Drive inventory by whole chip, skips already-published features, budgets **remaining TIFF copies + new NPY outputs + reserve**, and verifies each local TIFF copy with SHA-256. Missing S2 files remain missing and are zero-filled by the original preprocessing.

Preparation may take time because Drive must supply these inputs once. Rerunning resumes verified copies. Wait until it completes before starting extraction.


In [ ]:
subprocess.run([
    sys.executable, '-u', str(WORKFLOW), 'prepare',
    '--checkpoint', str(CHECKPOINT),
    '--train-features-dir', str(DRIVE_INPUTS),
    '--drive-output-dir', str(DRIVE_OUTPUTS),
    '--work-dir', str(WORK_DIR),
    '--num-shards', str(NUM_SHARDS), '--shard-id', str(SHARD_ID),
    '--copy-workers', str(COPY_WORKERS), '--reserve-gib', str(RESERVE_GIB),
    '--max-samples', str(MAX_SAMPLES),
], check=True)


## 4. Extract: local TIFF reads and local NPY writes
The plan supplies the exact selected sample IDs. The local subset is **not sharded again**. This phase uses only local inputs, local outputs, and the local checkpoint; no Google Drive resume scan runs during extraction. Existing valid local outputs are skipped.


In [ ]:
subprocess.run([
    sys.executable, '-u', str(WORKFLOW), 'extract',
    '--plan', str(PLAN),
    '--batch-size', str(BATCH_SIZE), '--num-workers', str(NUM_WORKERS),
    '--device', DEVICE,
], check=True)


## 5. Merge: publish to the same Drive directory and reclaim local space
This invokes repository code to copy/repair the five NPY files per chip/month, verify copied bytes with SHA-256, and check that the complete selected shard is available in Drive. `--remove-local` clears plan-owned local TIFFs and verified local NPYs only; original Drive TIFFs are never deleted.

If upload or verification fails, rerun the same shard's cells. Unverified local data is retained. After success, change `SHARD_ID` and repeat steps 3?5. Checkpoints and merged features remain on Drive; temporary `/content` files are disposable.


In [ ]:
subprocess.run([
    sys.executable, '-u', str(WORKFLOW), 'merge',
    '--plan', str(PLAN),
    '--remove-local',
], check=True)


## Optional: run all remaining shards sequentially
Run this instead of manually repeating steps 3?5. It stops immediately on an error and never stages the next shard before the previous one has been merged and cleaned. Leave enough free Google Drive quota for the complete cache.


In [ ]:
RUN_ALL_REMAINING = False  # Set True deliberately to start the loop.

if RUN_ALL_REMAINING:
    for shard_id in range(SHARD_ID, NUM_SHARDS + 1):
        plan = WORK_DIR / f'shard_{shard_id:03d}_of_{NUM_SHARDS:03d}' / 'plan.json'
        subprocess.run([
            sys.executable, '-u', str(WORKFLOW), 'prepare',
            '--checkpoint', str(CHECKPOINT), '--train-features-dir', str(DRIVE_INPUTS),
            '--drive-output-dir', str(DRIVE_OUTPUTS), '--work-dir', str(WORK_DIR),
            '--num-shards', str(NUM_SHARDS), '--shard-id', str(shard_id),
            '--copy-workers', str(COPY_WORKERS), '--reserve-gib', str(RESERVE_GIB),
            '--max-samples', str(MAX_SAMPLES),
        ], check=True)
        subprocess.run([
            sys.executable, '-u', str(WORKFLOW), 'extract', '--plan', str(plan),
            '--batch-size', str(BATCH_SIZE), '--num-workers', str(NUM_WORKERS), '--device', DEVICE,
        ], check=True)
        subprocess.run([
            sys.executable, '-u', str(WORKFLOW), 'merge', '--plan', str(plan), '--remove-local',
        ], check=True)
